# Phase 3 — Breeze-ASR-26 inference on KGI 90-query set

Runs **4 ablations** in one pass over 90 wav files:

| code | condition | initial_prompt |
|---|---|---|
| E1 | baseline | _(none)_ |
| E2 | + insurance terms | 凱基保險術語清單 |
| E3 | + PII pattern hint | 提示客戶會講姓名/身分證/生日/保單號/地址/電話 |
| E4 | E2 + E3 | 兩者合併 |

**環境**：Colab A100 (BF16, num_beams=1)，與之前 Phase 2 Breeze 報告同設定。

**輸入**：
- `audio/` (90 個 wav，16k mono，從 phase3_queries.csv 經 Yating TTS 產生)
- `data/phase3_queries.csv` (含 ref text)

**輸出**：`results/phase3_breeze_asr_results.csv` (90 row × {case_id, language, subtype, ref, hyp_e1..e4, dur_sec})

## 0. 環境檢查 + 安裝相依

In [ ]:
!nvidia-smi

In [ ]:
!pip install -q -U transformers==4.46.3 accelerate==1.1.1 librosa==0.10.2 soundfile==0.12.1 pandas tqdm

## 1. 掛 Drive + 設路徑

預設 Drive layout（請先把 `breeze_asr26/` 整包上傳到 `MyDrive/KGI_STT/`）：
```
/content/drive/MyDrive/KGI_STT/breeze_asr26/
├── audio/                    # 90 wav
├── data/phase3_queries.csv
└── results/                  # 本 notebook 輸出位置
```

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pathlib, os
BASE = pathlib.Path('/content/drive/MyDrive/KGI_STT/breeze_asr26')
AUDIO_DIR = BASE / 'audio'
CSV_PATH  = BASE / 'data' / 'phase3_queries.csv'
RESULTS_DIR = BASE / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
OUT_CSV = RESULTS_DIR / 'phase3_breeze_asr_results.csv'

assert AUDIO_DIR.exists(), AUDIO_DIR
assert CSV_PATH.exists(), CSV_PATH
wav_files = sorted(AUDIO_DIR.glob('*.wav'))
print(f'audio dir: {AUDIO_DIR}  ({len(wav_files)} wavs)')
print(f'csv      : {CSV_PATH}')
print(f'output   : {OUT_CSV}')

## 2. 載入 Breeze-ASR-26 (BF16)

In [ ]:
import torch
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline

MODEL_ID = 'MediaTek-Research/Breeze-ASR-26'
device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch_dtype = torch.bfloat16 if device == 'cuda' else torch.float32

print(f'loading {MODEL_ID}  device={device}  dtype={torch_dtype}')
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForSpeechSeq2Seq.from_pretrained(
    MODEL_ID, torch_dtype=torch_dtype, low_cpu_mem_usage=True
).to(device)
model.eval()

asr = pipeline(
    'automatic-speech-recognition',
    model=model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    chunk_length_s=30,           # Whisper 30s 手動分塊
    batch_size=1,
    torch_dtype=torch_dtype,
    device=device,
)
print('model ready')

## 3. 4 組 ablation 的 generate_kwargs

In [ ]:
# 凱基保險常見術語（含 Phase 2 hard cases 出現過的同音字錯誤關鍵詞）
INSURANCE_TERMS = (
    '凱基人壽、心康泰、新康泰、享安心、享放心、終身壽險、住院醫療附約、長期照顧保險、'
    '投資型保單、傷害險、意外險、健康險、儲蓄險、年金險、'
    '保價金、保單價值、解約金、滿期金、生存金、身故保險金、重大疾病、癌症、洗腎、'
    '巴氏量表、失能等級、加護病房、手術費用、診斷書、'
    '繳費、扣款、續保、停效、復效、變更、受益人、被保險人、要保人、保額、'
    '附約、主約、給付、理賠、申請、核保'
)

PII_HINT = (
    '客戶常會講出個人資料：姓名、身分證字號（一個英文字母加九個數字）、'
    '民國年份的出生年月日、保單號碼、家裡或公司地址、手機或市話號碼。'
)

ABLATIONS = {
    'e1': None,                                # baseline
    'e2': INSURANCE_TERMS,                     # + 保險詞
    'e3': PII_HINT,                            # + 個資模式
    'e4': INSURANCE_TERMS + ' ' + PII_HINT,    # 兩者合併
}

# 共用 generate_kwargs
BASE_GEN_KWARGS = dict(
    task='transcribe',
    language='zh',           # 強制中文輸出（含台→中字幕模式）
    num_beams=1,
    return_timestamps=False,
)

for k, v in ABLATIONS.items():
    print(f'{k}: {"(none)" if v is None else v[:60] + ("…" if len(v) > 60 else "")}')

## 4. 推論：每條 wav 跑 4 次

In [ ]:
import pandas as pd, librosa, time
from tqdm.auto import tqdm

queries = pd.read_csv(CSV_PATH).fillna('')
queries = queries.set_index('case_id')
print(f'loaded {len(queries)} queries from CSV')

# 斷點續跑：若已存在輸出 CSV，先讀進來，跳過已完成的 case_id
if OUT_CSV.exists():
    done_df = pd.read_csv(OUT_CSV).fillna('')
    done_ids = set(done_df['case_id'])
    rows = done_df.to_dict(orient='records')
    print(f'found existing results: {len(done_ids)} cases — will skip them')
else:
    done_ids, rows = set(), []

In [ ]:
def transcribe(audio_array, sr, prompt: str | None) -> str:
    gen_kwargs = dict(BASE_GEN_KWARGS)
    if prompt:
        # initial_prompt 透過 forced prompt_ids 注入
        prompt_ids = processor.get_prompt_ids(prompt, return_tensors='pt').to(device)
        gen_kwargs['prompt_ids'] = prompt_ids
    out = asr({'array': audio_array, 'sampling_rate': sr}, generate_kwargs=gen_kwargs)
    text = out['text'] if isinstance(out, dict) else out
    # 移除 prompt token 殘留（保險起見）
    if prompt and text.startswith(prompt):
        text = text[len(prompt):]
    return text.strip()

t0 = time.time()
for wav in tqdm(wav_files, desc='ASR x4 ablations'):
    case_id = wav.stem
    if case_id in done_ids:
        continue
    if case_id not in queries.index:
        print(f'WARN: {case_id} not in queries.csv, skip')
        continue

    audio, sr = librosa.load(str(wav), sr=16000, mono=True)
    dur = len(audio) / sr
    meta = queries.loc[case_id]
    row = {
        'case_id': case_id,
        'language': meta['language'],
        'subtype': meta['subtype'],
        'ref': meta['text'],
        'dur_sec': round(dur, 2),
    }
    for code, prompt in ABLATIONS.items():
        try:
            row[f'hyp_{code}'] = transcribe(audio, sr, prompt)
        except Exception as e:
            row[f'hyp_{code}'] = f'__ERROR__: {e}'
    rows.append(row)

    # 每 10 條存一次（防 Colab 斷線）
    if len(rows) % 10 == 0:
        pd.DataFrame(rows).to_csv(OUT_CSV, index=False)

pd.DataFrame(rows).to_csv(OUT_CSV, index=False)
print(f'done. {len(rows)} rows → {OUT_CSV}  elapsed={time.time()-t0:.1f}s')

## 5. 快速 sanity check

In [ ]:
df = pd.read_csv(OUT_CSV).fillna('')
print(f'rows: {len(df)}')
print(df['language'].value_counts().to_dict())
print(df['subtype'].value_counts().to_dict())

# 抽樣顯示
for _, r in df.sample(min(3, len(df)), random_state=42).iterrows():
    print('\n' + '=' * 72)
    print(f"{r['case_id']}  [{r['language']}/{r['subtype']}]  dur={r['dur_sec']}s")
    print(f"REF  : {r['ref']}")
    for code in ['e1', 'e2', 'e3', 'e4']:
        print(f"HYP_{code.upper()}: {r['hyp_' + code]}")

## 6. 把 CSV 也下載一份到本機（可選）

In [ ]:
from google.colab import files
files.download(str(OUT_CSV))